# Optional · Modern C++ from Python: RDataFrame and just-in-time compilation
We compare three ways to compute a quantity for $10^7$ events: a **Python loop**, **RDataFrame with a C++ expression** (JIT), and **RDataFrame multithreaded**.
Message: Python is the glue, C++ is the engine, and you do not need to leave the notebook to use either.

In [ ]:
import ROOT, time
import numpy as np

N = 2_000_000          # events (increase to 10_000_000 if your machine is fast)
def timeit(label, f):
    t0 = time.time(); r = f(); dt = time.time() - t0
    print("%-34s %7.2f s   result = %.6f" % (label, dt, r)); return dt

## 1 · Pure Python loop (do not do this)

In [ ]:
def python_loop(n=N // 20):             # 20x fewer events: the loop is that slow
    rng = np.random.default_rng(1)
    x = rng.normal(size=n); y = rng.normal(size=n)
    total = 0.0
    for a, b in zip(x, y):
        total += (a * a + b * b) ** 0.5 > 1.5
    return total / n

t_py = timeit("Python loop (N/20 events)", python_loop)
print("-> estimated time for all N events: %.1f s" % (t_py * 20))

## 2 · RDataFrame with a C++ function, compiled just in time

In [ ]:
ROOT.gInterpreter.Declare("""
double radius(double a, double b) { return std::sqrt(a * a + b * b); }
""")

def rdf_jit():
    df = ROOT.RDataFrame(N).Define("a", "gRandom->Gaus()").Define("b", "gRandom->Gaus()")
    sel = df.Define("r", "radius(a, b)").Filter("r > 1.5")
    return sel.Count().GetValue() / N

ROOT.DisableImplicitMT()
t_1 = timeit("RDataFrame, 1 thread, C++ JIT", rdf_jit)

## 3 · Same code, all cores

In [ ]:
ROOT.EnableImplicitMT()
t_mt = timeit("RDataFrame, implicit MT", rdf_jit)
print("threads:", ROOT.GetThreadPoolSize(), "  speed-up vs 1 thread: %.1f" % (t_1 / t_mt))

In [ ]:
def check(name, ok):
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name
check("RDataFrame is faster than the Python loop per event", t_1 / N < t_py / (N // 20))
exact = np.exp(-1.5 ** 2 / 2)           # P(r>1.5) for two independent standard normals
check("physics check: P(r>1.5) = exp(-1.125) = %.4f" % exact, abs(rdf_jit() - exact) < 0.005)

## Try this
1. Replace `radius` by a **lambda** defined in C++ with `auto`, and by a vectorised `ROOT::VecOps` expression on an `RVec`.
2. Time `df.Histo1D(...)` for 5 different histograms in one event loop vs 5 separate loops: `ROOT.RDF.RunGraphs`.
3. Write the same selection with numpy boolean masks. When is numpy competitive? When does RDataFrame win (memory, files, systematics with `Vary`)?